# 06 Oracle subset: all controllers on the same 28 days

The **oracle MPC** knows the exact future load and PV over its 144-step horizon. It is an **upper bound**, not a
controller that could be built. To keep its cost manageable, it was run on every 4th full validation day (28 days), each
day on its own with SoC reset to 0.5 at the start. All other controllers were run on exactly these days with the same
resets, so the numbers here differ slightly from the full validation run.

```{include} ../_caveats.md
```

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from common import *
plan = load("m4_oracle_subset/plan.json")
display(Markdown(provenance(plan, "m4_oracle_subset/plan.json")))
display(Markdown(f"Days: **{len(plan['days'])}** of {plan['n_full_validation_days']} full validation days, "
                 f"{plan['steps_per_regime']:,} steps per regime."))
order = [("no_battery", "no battery action"), ("droop_original", "droop, original"), ("droop_final_tuned", "droop, tuned"),
         ("persistence_mpc", "persistence MPC"), ("dpc_H144_dead_zone_30_70_seed_0", "DPC seed 0"),
         ("dpc_H144_dead_zone_30_70_seed_1", "DPC seed 1"), ("dpc_H144_dead_zone_30_70_seed_2", "DPC seed 2"),
         ("oracle_mpc_upper_bound", "oracle MPC (upper bound)")]
data = {(k, reg): load(f"m4_oracle_subset/{k}__{reg}.json") for k, _ in order for reg in REGIMES}
rows = []
for reg, rl in REGIMES.items():
    for k, label in order:
        r = data[(k, reg)]; s = r["summary"]
        rows.append({"regime": rl, "controller": label, "objective": fmt(r["objective"]),
                     "VVR": pct(s["vvr_share_of_steps_with_any_violation"], 3), "losses (kW)": fmt(s["mean_network_loss_kw"], 1),
                     "mean |P| (MW)": fmt(s["mean_abs_p_mw"], 4), "mean |Q| (Mvar)": fmt(s["mean_abs_q_mvar"], 3),
                     "end-of-day SoC": f"{fmt(r['end_of_day_soc_min'], 2)} - {fmt(r['end_of_day_soc_max'], 2)}",
                     "safety layer changed action": pct(r.get("share_battery_steps_safety_layer_changes_action"), 2)
                         if "share_battery_steps_safety_layer_changes_action" in r else "—"})
table(rows).set_index(["regime", "controller"])

*— means not applicable or not recorded for that controller. For DPC the safety layer is part of the policy itself, so no separate change of the action is recorded.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
colors = {"no battery action": INK_2, "droop, original": SERIES[4], "droop, tuned": SERIES[3], "persistence MPC": SERIES[1],
          "DPC seed 0": SERIES[0], "DPC seed 1": SERIES[0], "DPC seed 2": SERIES[0], "oracle MPC (upper bound)": SERIES[2]}
for ax, reg in zip(axes, REGIMES):
    vals = [data[(k, reg)]["objective"] for k, _ in order]
    ax.scatter(vals, range(len(order)), color=[colors[l] for _, l in order], s=50, zorder=3)
    ax.set_yticks(range(len(order)), [l for _, l in order]); ax.invert_yaxis(); ax.set_xscale("log")
    ax.set_xlabel("objective on the 28 days (log scale, lower is better)"); ax.set_title(REGIMES[reg], loc="left")
fig.tight_layout(); plt.show()

In [ ]:
o = {reg: data[("oracle_mpc_upper_bound", reg)] for reg in REGIMES}
display(Markdown("Oracle MPC: share of horizon steps where the forecast had to hold the last known value: " +
                 ", ".join(f"{REGIMES[r]} {pct(o[r]['share_held_forecast_horizon_steps'], 3)}" for r in REGIMES) +
                 ". Its action is changed by the safety layer on " +
                 ", ".join(f"{pct(o[r]['share_battery_steps_safety_layer_changes_action'], 2)} ({REGIMES[r]})" for r in REGIMES) +
                 " of battery steps, because its lossless SoC model differs from the real efficiency near the SoC limits."))

In [ ]:
import json
dpc_keys = [k for k, _ in order if k.startswith("dpc_")]
oracle_eod = [data[("oracle_mpc_upper_bound", reg)][k] for reg in REGIMES for k in ("end_of_day_soc_min", "end_of_day_soc_max")]
dpc_eod = [data[(k, reg)][e] for k in dpc_keys for reg in REGIMES for e in ("end_of_day_soc_min", "end_of_day_soc_max")]
soc_start = json.loads((ROOT / "configs" / "m3.json").read_text())["battery"]["soc_after_gap"]
display(Markdown(f"Every day starts at SoC {soc_start} (the reset value after a data gap in configs/m3.json). The oracle MPC ends the days at SoC "
                 f"{min(oracle_eod):.2f}-{max(oracle_eod):.2f}, while DPC ends them at {min(dpc_eod):.2f}-{max(dpc_eod):.2f}. "
                 "The oracle therefore also discharges more of the stored energy over each day; the stored results do not separate "
                 "how much of its advantage comes from this and how much from knowing the future."))
gap = []
for reg in REGIMES:
    dpc = float(np.median([data[(k, reg)]["objective"] for k in dpc_keys]))
    pers, orc = data[("persistence_mpc", reg)]["objective"], data[("oracle_mpc_upper_bound", reg)]["objective"]
    gap.append(f"{100 * (pers - dpc) / (pers - orc):.0f}% in the {REGIMES[reg]} regime")
display(Markdown("Descriptive, validation part only: of the objective gap between the persistence MPC and the oracle MPC, DPC "
                 "(median over seeds) closes " + " and ".join(gap) + "."))

**Reading (descriptive only):** on these validation days DPC lies between the persistence MPC and the oracle bound in both
regimes. This is one network, one untuned convex MPC, and the validation part, which was also used to select the DPC
policy; it is not a test result.